# IO List validation tool

## Setup

In [1]:
from collections import Counter
from pathlib import Path
from typing import List, Dict, Optional

import polars as pl
from IPython.core.display import HTML
from openpyxl import load_workbook, Workbook
from openpyxl.worksheet.worksheet import Worksheet
from polars import DataFrame

from zero_data.io_list import IOResult
from zero_data.io_list.readers.marpower import MarpowerReader


# Read in IO Lists

In [2]:
reader = MarpowerReader()

io_list_files = [p for p in Path("../io_lists").glob("*.xlsx")]
io_lists = {p.name: reader.read_io_list([p]) for p in io_list_files}
for name, io_list in io_lists.items():
    print(f"Loaded IO list: {name} with {len(io_list.topics)} topics")

/Users/floris/Library/Caches/pypoetry/virtualenvs/zero-data-9oE-VVSS-py3.13/lib/python3.13/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


Loaded IO list: 52422003_3210_AMCS IO-List R2.16.xlsx with 303 topics
Loaded IO list: 52422003_3211_PMS IO-List R2.7.xlsx with 3595 topics


## Check tag duplicates

Tags should be unique in an IO list. This analysis finds duplicate tags for each IO list

In [3]:
def check_tag_duplicates(io_result: IOResult):
    df = io_result.io_list
    tag_counts = df.group_by("tag").len("tag_count").sort("tag_count", descending=True)
    duplicates = tag_counts.filter(tag_counts["tag_count"] > 1)
    return duplicates


for name, io_list in io_lists.items():
    duplicates = check_tag_duplicates(io_list)
    result = f"<h2>{name} has {len(duplicates)} duplicate tags</h2><br><details><summary>Duplicate tags:</summary>"
    for row in duplicates.iter_rows(named=True):
        result = result + f"&emsp;{row['tag']} - Count: {row['tag_count']}<br>"
    result = result + "</details>"
    display(HTML(result))

## Schema For nested topics

If a topic is nested we expect the schema (i.e. the fields and their data types) to be the same for all topics with the same prefix. This analysis checks for topics with a given prefix and compares their schema to find any that do not match the most common schema.
Known nested topics are:
- AMCS:
    - power-tag
    - 450000 FIREDETECTION/

In [4]:
def get_non_matching_nested_topics(io_result: IOResult, topic_prefix: str)-> Optional[DataFrame]:
    nested_topic_df = io_result.io_list.filter(
        pl.col("mqtt_topic").str.starts_with(topic_prefix)
    )
    if len(nested_topic_df) == 0:
        return None

    nested_schema_df = (
        nested_topic_df.with_columns(
            pl.col("mqtt_json_path")
            .str.replace("$.", "", literal=True)
            .alias("mqtt_json_path")
        )
        .with_columns(
            pl.concat_str("mqtt_json_path", "data_type", separator=" ").alias(
                "field_and_type"
            )
        )
        .group_by(pl.col("mqtt_topic"))
        .agg("field_and_type")
    )
    most_common_schema = (
        nested_schema_df.group_by("field_and_type")
        .len("nr_of_topics")
        .sort("nr_of_topics", descending=True)
        .head(1)
        .to_dict(as_series=False)["field_and_type"][0]
    )
    common_schema_df = nested_schema_df.with_columns(
        (pl.col("field_and_type") == most_common_schema).alias("is_common_schema")
    )
    return common_schema_df


def summarize_schema_matching(schema_df: Optional[DataFrame], name: str):
    if schema_df is None:
        return HTML(f"&emsp;No topics with {name} found.")

    schema_match_dict = {
        k: v
        for (k, v) in schema_df.group_by("is_common_schema")
        .len("count")
        .rows(named=False)
    }

    result = f"""
    &emsp;Has {(schema_match_dict[True])} matching {name} topics
    &emsp;<details><summary>&emsp;Unmatched schemas:</summary>
    """
    unmatched_schemas = (
        schema_df.filter(~pl.col("is_common_schema"))
        .drop("is_common_schema")
        .rows(named=False)
    )
    for t, s in unmatched_schemas:
        result = result + f"&emsp;&emsp; {t} (has {len(s)} fields)<br>"
        for field in s:
            result = result + f"&emsp;&emsp;&emsp;{field}<br>"

    result = result + "</details>"
    return HTML(result)


amcs_io_lists = {k: v for (k, v) in io_lists.items()}

for name, io_list in amcs_io_lists.items():
    display(HTML(f"<h2>IO List: {name}</h2>"))
    schema_powertag_df = get_non_matching_nested_topics(io_list, "power-tag")
    display(summarize_schema_matching(schema_powertag_df, "Power Tag"))

    schema_fire_detection_df = get_non_matching_nested_topics(
        io_list, "450000 FIREDETECTION/"
    )
    display(summarize_schema_matching(schema_fire_detection_df, "Fire Detection"))

## Internal validation

The IO lists contain reference sheets with the possible values some columns can have. When these definitions are present they should be adhered to. This analysis gives a list of columns and their values that are not present in the validation lists. Two important (to us) columns are:
- System
- DataType

In [5]:
def headers(sheet: Worksheet) -> List[str]:
    return [c[0] for c in sheet.iter_cols(max_row=1) if c]


def iterate_excel_column(
    sheet: Worksheet, index: int, skip_extra_headers=0
) -> List[str]:
    return [
        c.value
        for c in next(
            sheet.iter_cols(
                min_col=index + skip_extra_headers, max_col=index, min_row=2
            )
        )
        if c and c.value
    ]


def get_definitions_from_workbook_tab(sheet: Worksheet) -> dict[str, List[str]]:
    _headers = headers(sheet)
    return {
        str(header.value): iterate_excel_column(sheet, header.col_idx)
        for header in _headers
    }


def get_definitions(workbook: Workbook) -> dict[str, List[str]]:
    project_definitions = get_definitions_from_workbook_tab(
        workbook["Project definitions"]
    )
    internal_lists = get_definitions_from_workbook_tab(workbook["Internal lists"])
    return internal_lists | project_definitions


def validate_column(
    sheet: Worksheet, column_name: str, expected_values: List[str]
) -> dict[str, int]:
    _headers = headers(sheet)
    column = [h for h in _headers if h.value == column_name]
    if len(column) == 0:
        raise Exception(f"Column {column_name} not found in sheet")
    col_idx = column[0].col_idx
    column_values = iterate_excel_column(sheet, col_idx, skip_extra_headers=0)
    invalid_values = [v for v in column_values if v not in expected_values]
    return dict(Counter(invalid_values))

In [6]:
def map_column_name(io_list_name: str) -> Dict[str, str]:
    """Column names are sometimes named differently in AMCS and PMS IO lists. This functions provides the mapping."""
    default = {"system": "System"}
    if "AMCS" in io_list_name:
        return {**default, "target_type": "Target Type"}
    elif "PMS" in io_list_name:
        return {**default, "target_type": "Target"}
    else:
        raise Exception(f"Unknown io_list_name: {io_list_name}")


def summarize_column_definition_validation(
    validation_dict: Dict[str, int], column_name: str
):
    result = f"<details><summary>&emsp;Column {column_name} values not matching definitions</summary>"
    for item, count in validation_dict.items():
        result = result + f"&emsp;&emsp;{item}: {count}\n"

    result = result + "</details>"
    return HTML(result)


for io_list_path in io_list_files:
    display(HTML(f"<h2>Validating IO list:{io_list_path.name}</h2>"))
    workbook = load_workbook(io_list_path, data_only=True)
    definitions = get_definitions(workbook)

    system_validation = validate_column(
        workbook["IO-List"], "System", definitions["Systems"]
    )
    display(summarize_column_definition_validation(system_validation, "System"))

    data_type_column = map_column_name(io_list_path.name)
    data_type_validation = validate_column(
        workbook["IO-List"], data_type_column["target_type"], definitions["Data Type"]
    )
    display(summarize_column_definition_validation(data_type_validation, "Data Type"))